## Parent-Document Retriever

 - 1. 작은 문서를 원하는 것:이렇게 되면 문서의 임베딩이 그 의미를 가장 정확하게 반영
 - 2. 각 청크의 맥락이 유지되도록 충분히 긴 문서를 원하는 경우
 - parent-Document 이 도구는 문서를 작은 조각으로 나누고 이조각을 관리. 검색을 진행할 경우엔 우선 이 작은 조각들을 찾아낸 다음, 이 조각들이 속한 원본 문서(또는 더 큰 조각)의 식별자(ID)를 통해 전체적인 맥락을 파악
 - 결론: 유사 문서의 부모 문서를 참고하므로, 조금 더 맥락을 담아 LLM에게 제공 가능

In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [ ]:
!pip install langchain==0.1.16

In [ ]:
from langchain.retrievers import ParentDocumentRetriever

In [ ]:
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
loaders = [
    PyPDFLoader("paper1.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load_and_split())

In [ ]:
model_name = 'jhgan/ko-sbert-nli'
encode_kwargs = {'normalize_embeddings':True}
ko_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs=encode_kwargs
)

In [ ]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

vectorstore = Chroma(
    collection_name = "full_documents", embedding_function=ko_embedding
)

store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore = vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

In [ ]:
retriever.add_documents(docs, ids=None)

In [ ]:
sub_docs = vectorstore.similarity_search("최저임금 인하")

In [ ]:
print("글 길이: {}\n\n".format(len(sub_docs[0].page_content)))
print(sub_docs[0].page_content)

글 길이: 496


2018~2019년 통합)추정계수(표준오차) P-value해석 17510.45(193.05)0.000***기준집단의 평균 시급 수준  1284.23(100.11)0.000***정책 시행 이후 전반적 임금수준이 평균 1,284원 상승-10080.43(293.39)0.000***최저임금 영향ㆍ미만 집단은 차상위임금집단보다 약 10,080원 낮음 -5863.54(550.97)0.000***비정규직은 정규직보다 평균 약 5,864원 낮음     3.71(273.03)0.989정규직 기준으로 정책 이후의 DID 효과는 유의하지 않음  -169.63(485.72)0.727정책 이후 비정규직의 공통 효과는 유의하지 않음  5574.43(680.95)0.000***사전 시점부터 처리집단 내 비정규직의 임금수준이 정규직보다 높음  -53.44(706.84)0.949삼중차분(DDD) 효과:최저임금 인상 후 비정규직의 임금격차 추가 축소효과는 통계적으로 유의하지 않음


In [ ]:
retrieved_docs = retriever.get_relevant_documents("최저임금 인하")

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


In [ ]:
print("글 길이: {}\n\n".format(len(retrieved_docs[0].page_content)))
print(retrieved_docs[0].page_content)

글 길이: 1087


  노동정책연구ㆍ2025년 제25권 제4호114나. 삼중차분분석<표 9>는 최저임금의 급격한 인상이 정규직과 비정규직 간 임금격차에 미친 영향을 삼중차분(Difference-in-Difference-in-Differences:DDD) 모형을 통해 분석한 결과를 제시한다. 본 분석은 단순히 정책 시행 전후의 변화뿐만 아니라, 처리집단(최저임금 영향집단)과 비교집단(차상위임금집단) 간의 차이를 추가적으로 고려함으로써, 최저임금 인상이 특정 집단(비정규직)에 미친 순수한 효과를 보다 정교하게 식별하고자 하였다.구체적으로, 분석모형에는 (1) 시기별 더미변수(정책 시행 전ㆍ후), (2) 고용형태 더미변수(정규직ㆍ비정규직), (3) 임금집단 더미변수(처리ㆍ비교집단)를 포함하고, 이 세 변수의 삼중 교호항(시기×고용형태×임금집단) 을 핵심 추정변수로 설정하였다. 이 교호항의 계수는 최저임금 인상이 정규직과 비정규직 간 임금격차에 미친 효과 중에서도, 최저임금의 영향을 직접적으로 받는 집단과 그렇지 않은 집단 간의 차별적 정책 효과를 계량적으로 나타낸다.<표 9> 삼중차분(DDD) 추정결과:최저임금 인상에 따른 고용형태별 임금격차 변화(2017년 대비 2018~2019년 통합)추정계수(표준오차) P-value해석 17510.45(193.05)0.000***기준집단의 평균 시급 수준  1284.23(100.11)0.000***정책 시행 이후 전반적 임금수준이 평균 1,284원 상승-10080.43(293.39)0.000***최저임금 영향ㆍ미만 집단은 차상위임금집단보다 약 10,080원 낮음 -5863.54(550.97)0.000***비정규직은 정규직보다 평균 약 5,864원 낮음     3.71(273.03)0.989정규직 기준으로 정책 이후의 DID 효과는 유의하지 않음  -169.63(485.72)0.727정책 이후 비정규직의 공통 효과는 유의하지 않음  5574.43(680.95)0.000***사전 시점부터 처리집단 내

## 본문의 Full_chunk가 너무 길때

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=800)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
vectorstore = Chroma(
    collection_name = "split_parents", embedding_function=ko_embedding
)

store = InMemoryStore()

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore = vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [ ]:
retriever.add_documents(docs)

In [ ]:
len(list(store.yield_keys()))

59

In [ ]:
sub_docs = vectorstore.similarity_search("최저임금 인하")

In [ ]:
print(sub_docs[0].page_content)

있다. 예컨대 근로시간 조정을 중심으로 한 국내 실증은 최저임금 인상 이후 일부 취약계층에서 월평균 소득이 근로시간 축소와 함께 조정될 수 있음을 보고한다(신우리 외, 2019). 이러한 국내외 근거를 종합하면, 최저임금의 효과는 (1) 임금 하단 압축과 분포개선, (2) 총고용보다는 근로시간ㆍ구성의 조정, (3) 기업 측 가격ㆍ이윤ㆍ투입대체를 통한
